# ចម្លើយលំហាត់ ជំពូក ២

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/exercise-solutions.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រម
- ## ១. ការស្វែងយល់ encode/decode របស់ tokenizer
- ## ២. ការកែ batch_size និង stride របស់ dataloader

## ០. ការត្រម

នាំចូលបណ្ណាល័យចាំបាច់សម្រាប់ចម្លើយលំហាត់។

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## ១. ការស្វែងយល់ encode/decode របស់ tokenizer

ពិនិត្យរបៀបដែល tiktoken បំបែក និងផ្គុំអត្ថបទឧទាហរណ៍ ព្រមទាំងផ្ទៀងផ្ទាត់លទ្ធផល។

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
integers = tokenizer.encode("Akwirw ier")
print(integers)

In [ ]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

In [ ]:
tokenizer.encode("Ak")

In [ ]:
tokenizer.encode("w")

In [ ]:
tokenizer.encode("ir")

In [ ]:
tokenizer.encode("w")

In [ ]:
tokenizer.encode(" ")

In [ ]:
tokenizer.encode("ier")

In [ ]:
tokenizer.decode([33901, 86, 343, 86, 220, 959])

## ២. ការកែ batch_size និង stride របស់ dataloader

សាកល្បងការកំណត់ dataloader ផ្សេងៗ ហើយពិនិត្យលទ្ធផល batches។

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # បំបែកអត្ថបទទាំងមូលជា token
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # ប្រើ sliding window ដើម្បីកាត់អត្ថបទជាបន្សំត្រួតគ្នា ប្រវែង max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128):
    # ចាប់ផ្តើម tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # បង្កើត dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # បង្កើត dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ